In [0]:
%run "../includes/librerias"

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
#Definimos parametros
v_esquema = "movie_gold"
v_tabla = "results_movie_genre_language"
v_partition = "file_date"
v_merge_condition = "target.movie_id = source.movie_id and target.language_id = source.language_id and target.genre_id = source.genre_id"

In [0]:
#DataFrames con la data con la cual se va a trabajar, incluye filtros y campos


##Tabla de carga completa
language_df = spark.read.table("movie_silver.languages")\
                        .select("language_id", "language_name")
##
genre_df = spark.read.table("movie_silver.genres")\
                     .select("genre_id", "genre_name")

#Tablas con data particionada
movies_df = spark.read.table("movie_silver.movies")\
                      .filter(
                               (col("file_date") == f"{v_file_date}")
                             )\
                      .select("movie_id", "title", "duration_time", "release_date", "vote_average", "year_release_date")
##
movie_languages_df = spark.read.table("movie_silver.movies_languages")\
                                .filter(
                                        (col("file_date") == f"{v_file_date}")
                                       )\
                               .select("movie_id", "language_id")
##
movie_genre_df = spark.read.table("movie_silver.movies_genres")\
                          .filter(
                                  (col("file_date") == f"{v_file_date}")
                                  )\
                           .select("movie_id", "genre_id")



In [0]:

movies_df = movies_df.filter(
                               (col("year_release_date") >= "2000")
                             )

#Join entre movie y movie_languages_df
movies_movie_languages_df = movies_df.join(movie_languages_df,
                                           movies_df.movie_id == movie_languages_df.movie_id
                                           , "inner")\
                                      .select(movies_df["*"], movie_languages_df.language_id)#\
                                      #.filter(movie_languages_df.movie_id.isNull())

#display(movies_movie_languages_df)

movies_languages_df = movies_movie_languages_df.join(language_df,
                                           movies_movie_languages_df.language_id == language_df.language_id
                                           , "inner")\
                                      .select(movies_movie_languages_df["*"], language_df.language_name)#\
                                      #.filter(language_df.language_id.isNull())

#display(movies_languages_df)



In [0]:
#Join entre movie y language

movies_movie_genre_df = movies_languages_df.join(movie_genre_df,
                                                 movies_languages_df.movie_id == movie_genre_df.movie_id
                                                ,"inner")\
                                           .select(movies_languages_df["*"], movie_genre_df.genre_id)#\
                                           #.filter(movie_genre_df.movie_id.isNull())
#hay datos nulos en genre
#display(movies_movie_genre_df)

movies_languages_genre_df = movies_movie_genre_df.join(genre_df,
                                                       movies_movie_genre_df.genre_id == genre_df.genre_id
                                                       , "inner")\
                                                 .select(movies_movie_genre_df["*"], genre_df.genre_name)#\
                                                 #.filter(movie_languages_df.movie_id.isNull())

#display(movies_languages_genre_df)


In [0]:
#Seleccionamos las columnas y ordenamos

results_movie_genre_language_df = movies_languages_genre_df.select( movies_languages_genre_df.movie_id,
                                                                    movies_languages_genre_df.language_id,
                                                                    movies_languages_genre_df.genre_id,
                                                                    movies_languages_genre_df.title,
                                                                    movies_languages_genre_df.duration_time,
                                                                    movies_languages_genre_df.release_date,
                                                                    movies_languages_genre_df.vote_average,
                                                                    movies_languages_genre_df.language_name,
                                                                    movies_languages_genre_df.genre_name
                                                                )\
                                                            .orderBy(movies_languages_genre_df.release_date.desc())


results_movie_genre_language_df = add_ingestion_date(results_movie_genre_language_df)
results_movie_genre_language_df = add_env(results_movie_genre_language_df)
final_df = add_file_date (results_movie_genre_language_df)

#display(results_movie_genre_language_df)

In [0]:
#Paso 5 - borramos la informacion de la tabla antes de cargarla
#overwrite_partition (v_esquema, v_tabla, v_partition, v_file_date)

In [0]:
resultado = merge_delta_lake (v_esquema, v_tabla, final_df, v_merge_condition, v_partition)
print(resultado)

In [0]:
#Guardamos en la capa gold 

#results_movie_genre_language_df.write.mode("append").partitionBy(v_partition).format("delta").saveAsTable(f"{v_esquema}.{v_tabla}")
#print(f"Se insertaron {results_movie_genre_language_df.count()} registros en la tabla {v_esquema}.{v_tabla}")


